# Graph-Based Fraud Detection

Wiki reference for [graph-based fraud detection](https://ml-viz-ruby.vercel.app/wiki/graph-fraud-detection).

**The idea in one sentence.** Fraud rings share devices, IPs, and payment methods, so building
an **entity graph** and **propagating risk** along its edges ("guilt by association") surfaces
accounts connected to known fraud — but the same propagation, pushed too hard, flags innocent
neighbours as **false positives**.

We build a GraphSAGE layer and a fraud-propagation rule from scratch, **validate that risk
spreads to connected accounts and stops at clean ones**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

## 1 — Building the entity-sharing graph

In [ ]:
# Simulated entity graph: 12 accounts, 5 shared devices
# (account_i, account_j) share a device
edges = [(0,1), (1,2), (2,3),   # chain: accounts 0-3 share devices
         (5,6), (6,7),           # separate cluster 5-7
         (9,10), (10,11)]        # another cluster

n_accounts = 12
adj = np.zeros((n_accounts, n_accounts))
for i,j in edges:
    adj[i,j] = adj[j,i] = 1.0

# Node features: [transaction_amount_z, account_age_days, n_countries_24h]
features = rng.uniform(0, 1, (n_accounts, 3))

# Known fraud labels (from human review): accounts 0 and 5
known_fraud = {0: 1.0, 5: 1.0}
fraud_scores = np.zeros(n_accounts)
for k, v in known_fraud.items():
    fraud_scores[k] = v

print("Graph adjacency matrix (12x12):")
print(adj.astype(int))
print("\nInitial fraud scores:", fraud_scores)

## 2 — GraphSAGE-style message passing

In [ ]:
def graphsage_mean_agg(h, adj, W_self, W_neigh):
    """
    One layer of GraphSAGE with mean aggregation.
    h: (N, d_in) node features
    adj: (N, N) adjacency (not normalized)
    W_self, W_neigh: (d_in, d_out) weight matrices
    """
    # Normalize adjacency: 1/degree per row
    deg = adj.sum(1, keepdims=True).clip(min=1)
    neigh_mean = (adj / deg) @ h          # mean of neighbors
    # Concatenate self + neighbor and project
    out = np.tanh(h @ W_self + neigh_mean @ W_neigh)
    return out

d_in, d_out = 3, 4
W_self  = rng.normal(0, 0.1, (d_in, d_out))
W_neigh = rng.normal(0, 0.1, (d_in, d_out))

h1 = graphsage_mean_agg(features, adj, W_self, W_neigh)
print(f"After 1 GraphSAGE layer: {features.shape} → {h1.shape}")
print("Node 1 representation (connected to known fraud node 0):", h1[1].round(3))
print("Node 9 representation (connected to fraud cluster 5-7):", h1[9].round(3))

### Validate: GraphSAGE produces one embedding per node

One GraphSAGE layer aggregates each node's own features with the **mean of its neighbours'**,
then projects — producing a fixed-size embedding per node that already blends local graph
structure. We confirm the output shape and that it is finite.

In [ ]:
print(f'features {features.shape} -> embeddings {h1.shape}')
assert h1.shape[0] == n_accounts, 'GraphSAGE outputs one embedding per node'
assert np.isfinite(h1).all(), 'the aggregated embeddings are finite'
print('\n✅ GraphSAGE blends each node with its neighbourhood into an embedding')

## 3 — Fraud score propagation

In [ ]:
def propagate_fraud_scores(scores, adj, alpha=0.4, n_steps=3):
    """
    Propagate fraud risk through the graph.
    alpha: weight given to neighbor scores vs own score.
    """
    s = scores.copy()
    for step in range(n_steps):
        deg = adj.sum(1).clip(min=1)
        neigh_avg = (adj @ s) / deg
        s = (1 - alpha) * s + alpha * neigh_avg
    return s

propagated = propagate_fraud_scores(fraud_scores, adj)
print("Account | Initial fraud | After graph propagation | Connected to fraud?")
for i in range(n_accounts):
    conn = any(adj[i,j]>0 and fraud_scores[j]>0 for j in range(n_accounts))
    print(f"  {i:3d}   |   {fraud_scores[i]:.3f}       |   {propagated[i]:.3f}                  | {'YES' if conn else 'no'}")

### Validate: risk propagates to connected accounts, not to clean ones

Accounts 0 and 5 are known fraud. Accounts sharing a device with them (e.g. 1, 2, 3 in the
0-chain) should gain risk after propagation, while a disconnected cluster (9-11) with no path to
fraud stays clean. We confirm both.

In [ ]:
print(f'account 1 (linked to fraud 0): {fraud_scores[1]:.3f} -> {propagated[1]:.3f}')
print(f'account 9 (no fraud path)   : {fraud_scores[9]:.3f} -> {propagated[9]:.3f}')
assert propagated[1] > fraud_scores[1], 'an account sharing a device with a fraudster gains risk'
assert propagated[9] < 0.01, 'an account with no path to fraud stays clean'
print('\n✅ guilt-by-association surfaces accounts connected to known fraud')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **over-propagation** | flags innocent neighbours — false positives (demo) |
| **guilt by association** | correlation, not proof; needs human review |
| **evolving graphs** | fraud rings restructure; the graph must be kept fresh |
| **cold nodes** | new accounts have no edges — graph signal is weak |
| **adversarial edges** | fraudsters can add benign edges to dilute their risk |

Demo: aggressive propagation inflates the risk of accounts far from real fraud.

In [ ]:
# The danger of guilt-by-association: pushed too hard (high alpha, many steps) it OVER-spreads
# risk, flagging accounts several hops from any real fraud — false positives that hit innocent
# users. Account 3 is three hops down the chain from fraud (0); aggressive propagation lights it
# up far more than the calibrated run. We confirm the false-positive inflation.
aggressive = propagate_fraud_scores(fraud_scores, adj, alpha=0.9, n_steps=10)
print(f'account 3 (3 hops from fraud): calibrated {propagated[3]:.3f}  vs aggressive {aggressive[3]:.3f}')
assert aggressive[3] > propagated[3], 'aggressive propagation inflates the risk of distant accounts (false positives)'
print('\nGuilt-by-association is powerful but over-propagation flags innocents -> tune alpha/steps and keep a human in the loop.')

## ✏️ Your turn — label propagation on bipartite graph

In [ ]:
def label_prop_step(scores, adj, n_steps=1):
    """
    Run n_steps of label propagation: each node averages its own score
    with the average of its neighbors' scores.
    Returns updated scores.
    """
    # TODO(you): implement n_steps iterations of the update:
    # s_new[v] = 0.5 * s[v] + 0.5 * mean(s[neighbors of v])
    return ...

prop = label_prop_step(fraud_scores, adj, n_steps=2)
print("After 2-step label propagation:")
for i in range(n_accounts):
    print(f"  Account {i:2d}: {prop[i]:.4f}")

<details><summary>Solution</summary>

```python
def label_prop_step(scores, adj, n_steps=1):
    s = scores.copy()
    for _ in range(n_steps):
        deg = adj.sum(1).clip(min=1)
        s = 0.5 * s + 0.5 * (adj @ s) / deg
    return s
```
</details>

## Key takeaways

- **Entity graphs expose fraud rings:** shared devices/IPs connect colluding accounts.
- **GraphSAGE** embeds each node with its neighbourhood (verified).
- **Risk propagation** surfaces accounts connected to known fraud, and stops at clean ones
  (verified).
- **Over-propagation causes false positives:** distant innocents get flagged (demo) — tune
  aggressiveness and keep human review.